# TSV Concentric · Q3D model inspection

Build a Concentric Signal joined through a hollow TSV, with separate top
and bottom Ground planes connected by four Ground TSVs. This notebook
inspects the Q3D model and Net assignments; it does **not** extract a
capacitance matrix. See the [TSV model-inspection
guide](../../../../simulation/tsv-model-inspection.qmd) and [capacitance
requirement](../../../../simulation/requirements/electrostatic-capacitance.qmd).

Use this OrPen source candidate with the fixed [SCGSim
checkpoint](https://github.com/OrPenStrike/scgsim/tree/c9270c7e76e0b2df187dddde462079e00d410dd0)
in an isolated Python 3.12/3.13 environment. AEDT 2024 R2 and the
checkpoint’s PyAEDT 1.3.0 cohort are needed only for the explicit native
transaction. Ordinary project dependency pins are not changed by this
notebook.

## Setup and Imports

The following cells use public OrPen source and SCGSim models. No
Desktop opens during imports or source authoring.

## Setup and Run Controls

`source` constructs the layout and source records only.
`prepare_handoff` creates a fresh backend input package; `build_model`
additionally opens an owned Desktop, builds, saves and releases the
model. `inspect_model` reads only an explicit returned run in a fresh
kernel. No action here performs Analyze.

In [ ]:
import hashlib
import json
import subprocess
import sys
from dataclasses import replace
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

from IPython.display import display
from scgsim.aedt import PdkMaterial, prepare_q3d_from_geometry, resolve_prepared_model
from scgsim.geometry import (
    GeometryBuildInput,
    GeometryPlanSnapshot,
    LayoutPolygonSpec,
    SemanticEntitySpec,
)

import orpen_sc_pdk
from orpen_sc_pdk.cells import get_tsv_source_records, tsv_concentric_signal_coupon

WORKFLOW_ACTION = "source"  # source | prepare_handoff | build_model | inspect_model
DIE = "D0"
DIE_SIDE_UM = 1000.0
BORE_DIAMETER_UM = 60.0
WALL_THICKNESS_UM = 0.2
SIGNAL_OUTER_RADIUS_UM = 100.0
GAP_UM = 20.0
GROUND_TSV_OFFSET_UM = 300.0
MODEL = "thin_film"  # face films become sheets; TSV levels remain retained_solid.
RUN_CONTROL = None
RUN_ID = f"tsv-concentric-q3d-{uuid4().hex[:12]}"
OUTPUT_ROOT = Path(".artifacts/SCGSimExamples/TSV")
RUN_DIR = OUTPUT_ROOT / RUN_ID
SOURCE_DIR = OUTPUT_ROOT / f"{RUN_ID}-source"
RETURNED_RUN_DIR = None  # Set to the actual returned run when selecting inspect_model.
PROJECT_NAME = "TSVConcentricQ3DThinFilmModelOnly"
DESIGN_NAME = "Q3D"
REGION_PADDING_XYZ_UM = (250.0, 250.0, 1000.0)
RESOURCES = None  # No solver HPC request; model-process allocation is arranged separately.
SIGNAL_ENTITY_IDS = (f"{DIE}/TOP_SIGNAL", f"{DIE}/BOTTOM_SIGNAL", f"{DIE}/SIGNAL_WALL")
GROUND_ENTITY_IDS = (f"{DIE}/TOP_GROUND", f"{DIE}/BACK_GROUND") + tuple(
    f"{DIE}/GROUND_TSV_{i}" for i in range(4)
)
orpen_sc_pdk.activate()
display({name: version(name) for name in ("orpen-sc-pdk", "gdsfactory", "scgsim")})

Solver resource flags are not an aggregate OS cap or CAD containment
mechanism. Execute a native action only under the available host
allocation. Q3D has no HFSS port or radiation boundary in this case.
Source/Sink terminals are not added to this terminal-free C/G model.

## Create Simulation Component / Coupon

In [ ]:
if WORKFLOW_ACTION != "inspect_model":
    coupon = tsv_concentric_signal_coupon(
        die=DIE,
        die_side_um=DIE_SIDE_UM,
        bore_diameter_um=BORE_DIAMETER_UM,
        wall_thickness_um=WALL_THICKNESS_UM,
        signal_outer_radius_um=SIGNAL_OUTER_RADIUS_UM,
        gap_um=GAP_UM,
        ground_tsv_offset_um=GROUND_TSV_OFFSET_UM,
    )
    coupon.plot()

M1 Al thickness and the 500 µm Silicon substrate come from the PDK. The
0.2 µm TiN wall is a finite solid; its interior remains empty. These are
nominal OrPen research dimensions, not a reproduction of a cited
fabrication process.

## Initialize AEDT Project / App

The selected design is Q3D. The native CLI owns initialization later
during `build_model`; no manual Desktop attachment or second client is
used here.

## Import GDS and Build the HFSS/Q3D/Q2D Model

The canonical OrPen getter supplies actual placed polygon exteriors,
holes, source occurrences, physical levels and materials. This cell only
adopts the explicit case Net grouping and constructs SCGSim’s public
input records. It does not infer connectivity or register invented
children. This cell constructs source input; actual native import/build
belongs to the later CLI transaction, not this source observation.

In [ ]:
if WORKFLOW_ACTION != "inspect_model":
    bounds = coupon.dbbox()
    regions = coupon.info["tsv_source"]["source_regions"]
    source_bottom = min(r["z_interval_um"][0] for r in regions)
    source_top = max(r["z_interval_um"][1] for r in regions)
    px, py, pz = REGION_PADDING_XYZ_UM
    background = {
        "semantic_id": "CALLER_BACKGROUND",
        "role": "solution_region",
        "material_id": "vacuum",
        "material_kind": "vacuum",
        "priority": 99,
        "geometry_kind": "domain",
        "geometry": {
            "domain_bounds_um": {
                "x_min_um": bounds.left - px,
                "x_max_um": bounds.right + px,
                "y_min_um": bounds.bottom - py,
                "y_max_um": bounds.top + py,
            },
            "z_min_um": source_bottom - pz,
            "z_max_um": source_top + pz,
        },
        "metadata": {"meaning": "computational background complement"},
    }
    source = get_tsv_source_records(coupon, host_solution_region=background)
    caller_nets = {
        **dict.fromkeys(SIGNAL_ENTITY_IDS, "Signal"),
        **dict.fromkeys(GROUND_ENTITY_IDS, "Ground"),
    }
    records = source["geometry_input"]
    stack = source["stack"]
    for layer in stack["layers"]:
        layer["net_id"] = caller_nets[layer["semantic_id"]]
    # Keep physical material records; add only the compiler's kind vocabulary.
    geometry_kinds = {
        "superconductor": "conductor",
        "dielectric": "dielectric",
        "vacuum": "vacuum",
    }
    stack["materials"] = {
        name: {
            **record,
            "kind": geometry_kinds[record["material_kind"]],
            "permittivity": record["relative_permittivity"],
        }
        for name, record in stack["materials"].items()
    }
    build_input = GeometryBuildInput(
        polygons=tuple(LayoutPolygonSpec(**p) for p in records["polygons"]),
        entities=tuple(
            SemanticEntitySpec(**{**e, "net_id": caller_nets.get(e["semantic_id"])})
            for e in records["entities"]
        ),
        solution_regions=records["solution_regions"],
        metadata={**records["metadata"], "caller_final_nets": caller_nets},
    )
    display(
        {
            "source_parameters": coupon.info["tsv_source"]["parameters"],
            "source_contacts": records["metadata"]["source_contacts"],
            "source_ports": records["metadata"]["source_ports"],
            "caller_nets": caller_nets,
            "modeling_declarations": {
                e.semantic_id: e.metadata.get("aedt_modeling")
                for e in build_input.entities
                if e.material_kind == "conductor"
            },
        }
    )

## Geometry Verification

These are source observations. Silicon already has the authored bore
holes; the TSV wall is an annulus, not a filled cylinder. In
`thin_film`, face films follow their declared contact side while TSV
`retained_solid` levels stay finite. SCGSim owns the effective-Z mapping
and the actual backend geometry. The current AEDT path consumes sampled
GDS polygon witnesses; exact circular recipes remain source intent, not
active native curve reconstruction.

### Fresh source witness

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "build_model"}:
    SOURCE_DIR.mkdir(parents=True, exist_ok=False)
    gds_path = coupon.write_gds(SOURCE_DIR / "source.gds", with_metadata=False)
    gds_bytes = Path(gds_path).read_bytes()
    gds_sha256 = hashlib.sha256(gds_bytes).hexdigest()
    stack["metadata"]["gds_sha256"] = gds_sha256
    build_input = replace(build_input, metadata={**build_input.metadata, "gds_sha256": gds_sha256})

The exported witness and normalized source records come from the same
component. The backend helper owns actual lowering; this notebook does
not interpret a source GDS hash as an actual native geometry
observation.

## Materials and Boundaries

Al and TiN remain the physical source materials; their PDK
superconducting declarations select ideal native conductor treatment.
Silicon is a perforated dielectric solid. The caller-authored background
is the surrounding complement, not a second overlapping occupied body.
Final Nets are the explicit grouping above, not a connectivity result
inferred by the notebook.

## Simulation Setup

`RUN_CONTROL = None` requests model-only preparation. No frequency,
modes, adaptive setup or sweep is silently filled in. A later physics
calculation needs its own explicitly authored setup and operation
assignment.

## Simulation Configuration

In [ ]:
HANDOFF = None
if WORKFLOW_ACTION in {"prepare_handoff", "build_model"}:
    # Public detached record, not a claim that GeometryPlan.prepare registered these children.
    snapshot = GeometryPlanSnapshot(
        build_input, json.dumps(stack), gds_bytes, gds_sha256, tuple(source["source_occurrences"])
    )
    used_materials = {e.material_id for e in build_input.entities}
    materials = {
        name: PdkMaterial(
            material_id=name,
            kind=stack["materials"][name]["material_kind"],
            is_superconducting=stack["materials"][name]["is_superconducting"],
            library_name=stack["materials"][name]["aedt_library_name"],
        )
        for name in used_materials
    }
    HANDOFF = prepare_q3d_from_geometry(
        snapshot,
        modeling=MODEL,
        output_dir=RUN_DIR,
        project_name=PROJECT_NAME,
        design_name=DESIGN_NAME,
        materials=materials,
        net_types={"Signal": "Signal", "Ground": "Ground"},
        physical_ground_nets=("Ground",),
        run_control=RUN_CONTROL,
        region_padding_um=(px, px, py, py, pz, pz),
        resources=RESOURCES,
    )
    display(HANDOFF)

The detached snapshot binds the freshly exported GDS to the same source
occurrences and holes. Physical Al/TiN identities are retained; native
ideal superconductor treatment follows the PDK material declaration.
`None` means there is no Analysis Setup, extraction frequency or
adaptive-pass selection.

## Solve and Export

**Model-only: no solve.** The explicit CLI action builds, reads, saves
and releases the prepared model; it never calls Analyze in this
notebook.

In [ ]:
if WORKFLOW_ACTION == "build_model":
    subprocess.run(
        [
            sys.executable,
            "-m",
            "scgsim.aedt.run",
            "--handoff",
            str(HANDOFF.metadata_path),
            "--execute",
            "--prepare-only",
        ],
        check=True,
    )

## Adaptive-Pass Convergence / Solver Diagnostics

No adaptive pass or convergence history is expected. Keep actual native
preparation errors and save/release failures visible instead of
retrying.

## Results: Plots and Readable Tables

### Physics Analysis Results

No C/G matrix, field energy or electrical-continuity result is expected.
Inspect actual sheet/solid membership, Silicon bores, material and
Signal/Ground assignments. Empty or inconclusive contact queries remain
unknown.

In [ ]:
PREPARED_MODEL = None
if WORKFLOW_ACTION == "inspect_model":
    PREPARED_MODEL = resolve_prepared_model(RETURNED_RUN_DIR)
    display(PREPARED_MODEL.project_path)
    display(PREPARED_MODEL.readback)

### Simulation Performance / Benchmarks

There are no adaptive passes or solved benchmarks. Preserve
preparation/save/ release provenance from the returned package; a
successful model build is not capacitance accuracy or convergence
evidence.

In [ ]:
if PREPARED_MODEL is not None:
    display({"mode": PREPARED_MODEL.mode, "model_stage": PREPARED_MODEL.model_stage})

## Save and Release AEDT

The public Prepare Only transaction saves and releases its own Desktop.
Do not open a second client to release it. Keep `.aedt` and its required
unsolved companions together; use a separate writable copy for Human
edits rather than modifying the hash-bound returned producer project.
Old Solid models and images remain historical, not outputs of this Thin
Film notebook.